# FINAL SMART LOCK SYSTEM

המחברת מותאמת לעבודה ב־Jupyter.  
התא הבא מגדיר את כל המערכת אך אינו מפעיל אותה.  
להפעלה, יש להריץ לאחר מכן את תא ההפעלה האחרון.


In [ ]:
# ============================================================
# FINAL SMART LOCK SYSTEM - PYNQ-Z2
# Face Recognition + Keypad Backup + PIR + Light Sensor + MC-38 Door Sensor
# Relay + Stepper Motor + LEDs + Buzzer
# Keypad Wrong Code = 1 Failed Attempt
# 3 Wrong Keypad Codes = Capture Photo + Telegram + Lockout
# 3 Failed Face Recognition Events = 1 Failed Attempt + Telegram
# Telegram Alerts + Photo Sending + CSV Logs + Lockout
# ============================================================

from pynq import Overlay
from pynq.lib import AxiGPIO

import cv2
import json
import time
import csv
import queue
import threading
import requests
from datetime import datetime
from enum import Enum
from pathlib import Path


# ============================================================
# 1. Project Configuration
# ============================================================

BITSTREAM_FILE = "design_1.bit"


AUTHORIZED_USER_ID = 1
CONFIDENCE_THRESHOLD = 60.0

TRAINER_PATH = "trainer.yml"

CASCADE_PATHS = [
    "haarcascade_frontalface_default.xml",
    "/usr/share/opencv4/haarcascades/haarcascade_frontalface_default.xml",
    "/home/xilinx/jupyter_notebooks/base/video/data/haarcascade_frontalface_default.xml"
]

CAPTURED_IMAGE_PATH = "smart_lock_captured.jpg"
RESULT_IMAGE_PATH = "smart_lock_result.jpg"
UNKNOWN_IMAGE_PATH = "unknown_person.jpg"
KEYPAD_INTRUDER_IMAGE_PATH = "keypad_intruder.jpg"

LOG_FILE = "smart_lock_access_log.csv"


# ============================================================
# 3. System Constants
# ============================================================

DARK_VALUE = 1

BUZZER_FREQ_HZ = 3000

KEYPAD_BEEP_SEC = 0.035
KEYPAD_DEBOUNCE_SEC = 0.18

MOTOR_STOP  = 0b000
MOTOR_OPEN  = 0b001
MOTOR_CLOSE = 0b010
MOTOR_RESET = 0b100

MOTOR_RUN_TIME_SEC = 4.096

# Door-controlled automatic locking
DOOR_OPEN_TIMEOUT_SEC = 15.0
DOOR_CLOSE_TIMEOUT_SEC = 60.0
DOOR_POLL_INTERVAL_SEC = 0.05
DOOR_STABLE_TIME_SEC = 0.20
DOOR_LOCK_DELAY_SEC = 3.0

RELAY_PRE_LIGHT_SEC = 1.0
MOTION_COOLDOWN_SEC = 3.0

MAX_FAILED_ATTEMPTS = 3
MAX_FAILED_FACE_RECOGNITION = 3
LOCKOUT_TIME_SEC = 30


# ============================================================
# 4. Peripheral Bit Mapping
# ============================================================

PIR_BIT = 0
LIGHT_SENSOR_BIT = 1
DOOR_SENSOR_BIT = 2

# Verified MC-38 logic with NC contact and 10 kOhm pull-up:
# 0 = door closed, 1 = door open or disconnected wire
DOOR_CLOSED_VALUE = 0

MOTION_LED_BIT = 0
BUZZER_BIT = 1
UNLOCKED_LED_BIT = 2
LOCKED_LED_BIT = 3
RELAY_BIT = 4


# ============================================================
# 5. State Machine
# ============================================================

class SmartLockState(Enum):
    locked = "LOCKED"
    motor_opening = "MOTOR_OPENING"
    waiting_for_door_open = "WAITING_FOR_DOOR_OPEN"
    holding_door_open = "HOLDING_DOOR_OPEN"
    waiting_for_door_close = "WAITING_FOR_DOOR_CLOSE"
    locking_delay = "LOCKING_DELAY"
    motor_closing = "MOTOR_CLOSING"


# ============================================================
# 6. General Utility Functions
# ============================================================

def now_string():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def bit_is_set(value, bit_index):
    return ((value >> bit_index) & 1) == 1


def set_bit(value, bit_index):
    return value | (1 << bit_index)


def clear_bit(value, bit_index):
    return value & ~(1 << bit_index)


# ============================================================
# 7. Smart Lock Runtime
# ============================================================

class SmartLockSystem:
    def __init__(self):
        # __file__ exists when running as a .py script, but not inside Jupyter.
        if "__file__" in globals():
            self.base_dir = Path(__file__).resolve().parent
        else:
            self.base_dir = Path.cwd()

        self.overlay = None

        self.motor_gpio = None
        self.peripherals_gpio = None
        self.keypad_gpio = None

        self.motor_channel = None
        self.inputs_channel = None
        self.outputs_channel = None
        self.keypad_channel = None

        self.motor_mask = None
        self.inputs_mask = None
        self.outputs_mask = None
        self.keypad_mask = None

        self.output_state = 0

        self.failed_attempts = 0
        self.failed_face_recognition_count = 0
        self.lockout_until = 0

        self.face_detector = None
        self.face_recognizer = None

        self.last_keypad_key = None
        self.last_keypad_time = 0
        self.keypad_buffer = ""

        self.state = SmartLockState.locked
        self.motor_deadline = 0
        self.active_motor_command = MOTOR_STOP
        self.close_sequence = []
        self.close_step_index = 0

        self.door_wait_target = None
        self.door_wait_started_at = 0
        self.door_wait_timeout = 0
        self.door_wait_stable_time = DOOR_STABLE_TIME_SEC
        self.door_wait_stable_since = None
        self.door_wait_last_value = None
        self.hold_open_until = 0
        self.lock_delay_until = 0

        config_path = self.runtime_path("smart_lock.local.json")
        try:
            with config_path.open(encoding="utf-8") as config_file:
                config = json.load(config_file)
        except FileNotFoundError as exc:
            raise FileNotFoundError(
                "Create smart_lock.local.json using the README instructions."
            ) from exc

        if not isinstance(config, dict):
            raise ValueError("smart_lock.local.json must contain a JSON object.")

        self.password = config.get("keypad_code")
        if not isinstance(self.password, str) or not (
            1 <= len(self.password) <= 8
            and self.password.isascii()
            and self.password.isdigit()
        ):
            raise ValueError("Set keypad_code to a string of 1 to 8 digits.")

        self.telegram_bot_token = config.get("telegram_bot_token", "")
        self.telegram_chat_id = config.get("telegram_chat_id", "")
        if not all(isinstance(value, str) for value in (
            self.telegram_bot_token, self.telegram_chat_id
        )):
            raise ValueError("Telegram settings must be JSON strings.")
        self.telegram_enabled = bool(self.telegram_bot_token and self.telegram_chat_id)
        self.telegram_queue = queue.Queue(maxsize=20)
        self.telegram_worker = threading.Thread(
            target=self._telegram_worker,
            name="smart_lock_telegram_worker",
            daemon=True
        )

        self.telegram_worker.start()

    # ========================================================
    # Paths
    # ========================================================

    def runtime_path(self, path_value):
        path = Path(path_value)

        if path.is_absolute():
            return path

        return self.base_dir / path

    # ========================================================
    # Logging
    # ========================================================

    def init_log_file(self):
        log_path = self.runtime_path(LOG_FILE)

        if not log_path.exists():
            with log_path.open(mode="w", newline="") as file:
                writer = csv.writer(file)
                writer.writerow([
                    "Timestamp",
                    "Source",
                    "Result",
                    "Confidence",
                    "FailedAttempts",
                    "FaceFailedCount",
                    "SystemState",
                    "ImagePath"
                ])

    def write_log(self, source, result, confidence="", system_state="LOCKED", image_path=""):
        log_path = self.runtime_path(LOG_FILE)

        with log_path.open(mode="a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([
                now_string(),
                source,
                result,
                confidence,
                self.failed_attempts,
                self.failed_face_recognition_count,
                system_state,
                image_path
            ])

    # ========================================================
    # Telegram Queue
    # ========================================================

    def send_telegram_message(self, message):
        if not self.telegram_enabled:
            return

        self.enqueue_telegram_task({
            "kind": "message",
            "message": message
        })

    def send_telegram_photo(self, image_path, caption=""):
        if not self.telegram_enabled:
            return

        image_file = self.runtime_path(image_path)

        if not image_file.exists():
            print("Telegram photo was not sent. Image does not exist:", image_path)
            return

        self.enqueue_telegram_task({
            "kind": "photo",
            "image_path": str(image_file),
            "caption": caption
        })

    def enqueue_telegram_task(self, task):
        try:
            self.telegram_queue.put_nowait(task)
        except queue.Full:
            print("Telegram queue is full. Alert was dropped.")

    def _telegram_worker(self):
        while True:
            task = self.telegram_queue.get()

            try:
                if not self.telegram_enabled:
                    continue

                if task.get("kind") == "message":
                    self._send_telegram_message_now(task.get("message", ""))
                elif task.get("kind") == "photo":
                    self._send_telegram_photo_now(
                        task.get("image_path", ""),
                        task.get("caption", "")
                    )

            finally:
                self.telegram_queue.task_done()

    def _send_telegram_message_now(self, message):
        try:
            url = "https://api.telegram.org/bot{}/sendMessage".format(self.telegram_bot_token)

            data = {
                "chat_id": self.telegram_chat_id,
                "text": message
            }

            response = requests.post(url, data=data, timeout=10)

            if response.status_code != 200:
                print("Telegram message failed (HTTP status):", response.status_code)

        except Exception as e:
            print("Telegram message error:", type(e).__name__)

    def _send_telegram_photo_now(self, image_path, caption=""):
        image_file = Path(image_path)

        if not image_file.exists():
            print("Telegram photo was not sent. Image does not exist:", image_path)
            return

        try:
            url = "https://api.telegram.org/bot{}/sendPhoto".format(self.telegram_bot_token)

            with image_file.open("rb") as file:
                files = {
                    "photo": file
                }

                data = {
                    "chat_id": self.telegram_chat_id,
                    "caption": caption
                }

                response = requests.post(
                    url,
                    data=data,
                    files=files,
                    timeout=20
                )

            if response.status_code != 200:
                print("Telegram photo failed (HTTP status):", response.status_code)

        except Exception as e:
            print("Telegram photo error:", type(e).__name__)

    # ========================================================
    # Overlay and GPIO Initialization
    # ========================================================

    def load_system_overlay(self):
        print("Loading overlay...")

        self.overlay = Overlay(str(self.runtime_path(BITSTREAM_FILE)))

        print("Overlay loaded successfully.")
        print("Available IP blocks:")

        for ip_name in self.overlay.ip_dict.keys():
            print("-", ip_name)

        self.motor_gpio = AxiGPIO(self.overlay.ip_dict["motor_control_gpio"])
        self.peripherals_gpio = AxiGPIO(self.overlay.ip_dict["peripherals_gpio"])
        self.keypad_gpio = AxiGPIO(self.overlay.ip_dict["keypad_status_gpio"])

        self.motor_channel = self.motor_gpio.channel1
        self.inputs_channel = self.peripherals_gpio.channel1
        self.outputs_channel = self.peripherals_gpio.channel2
        self.keypad_channel = self.keypad_gpio.channel1

        self.motor_mask = self.confirm_gpio_mask(
            ip_name="motor_control_gpio",
            channel_number=1,
            channel=self.motor_channel,
            channel_name="motor_control_gpio.channel1"
        )
        self.inputs_mask = self.confirm_gpio_mask(
            ip_name="peripherals_gpio",
            channel_number=1,
            channel=self.inputs_channel,
            channel_name="peripherals_gpio.channel1"
        )
        self.outputs_mask = self.confirm_gpio_mask(
            ip_name="peripherals_gpio",
            channel_number=2,
            channel=self.outputs_channel,
            channel_name="peripherals_gpio.channel2"
        )
        self.keypad_mask = self.confirm_gpio_mask(
            ip_name="keypad_status_gpio",
            channel_number=1,
            channel=self.keypad_channel,
            channel_name="keypad_status_gpio.channel1"
        )

        print("GPIO blocks initialized successfully.")

    def confirm_gpio_mask(self, ip_name, channel_number, channel, channel_name):
        width = self.get_hwh_gpio_width(ip_name, channel_number)

        if width is None:
            for attribute_name in ("length", "width", "_width"):
                width_value = getattr(channel, attribute_name, None)

                if isinstance(width_value, int) and width_value > 0:
                    width = width_value
                    break

        if width is None or width <= 0:
            raise RuntimeError("Could not confirm GPIO width for " + channel_name)

        mask = (1 << width) - 1
        print("GPIO mask confirmed for {}: width = {}".format(channel_name, width))
        return mask

    def get_hwh_gpio_width(self, ip_name, channel_number):
        ip_info = self.overlay.ip_dict.get(ip_name, {})
        parameters = ip_info.get("parameters", {})

        if channel_number == 1:
            keys = ["C_GPIO_WIDTH", "GPIO_WIDTH"]
        else:
            keys = ["C_GPIO2_WIDTH", "GPIO2_WIDTH"]

        for key in keys:
            if key in parameters:
                try:
                    return int(str(parameters[key]), 0)
                except Exception:
                    return None

        return None

    def init_outputs(self):
        self.output_state = 0
        self.write_outputs()

        self.set_locked_led(True)
        self.set_unlocked_led(False)
        self.set_motion_led(False)
        self.set_relay(False)
        self.set_buzzer(False)

        self.motor_write(MOTOR_STOP)

        print("Outputs initialized. System state: LOCKED")

    # ========================================================
    # GPIO Read / Write
    # ========================================================

    def write_outputs(self):
        self.outputs_channel.write(self.output_state, self.outputs_mask)

    def set_output_bit(self, bit_index, state):
        if state:
            self.output_state = set_bit(self.output_state, bit_index)
        else:
            self.output_state = clear_bit(self.output_state, bit_index)

        self.write_outputs()

    def read_inputs(self):
        return self.inputs_channel.read() & self.inputs_mask

    def read_pir(self):
        value = self.read_inputs()
        return 1 if bit_is_set(value, PIR_BIT) else 0

    def read_light_sensor(self):
        value = self.read_inputs()
        return 1 if bit_is_set(value, LIGHT_SENSOR_BIT) else 0

    def read_door_sensor(self):
        value = self.read_inputs()
        return 1 if bit_is_set(value, DOOR_SENSOR_BIT) else 0

    def is_door_closed(self):
        return self.read_door_sensor() == DOOR_CLOSED_VALUE

    def get_door_state(self):
        return "CLOSED" if self.is_door_closed() else "OPEN"

    def is_dark(self):
        return self.read_light_sensor() == DARK_VALUE

    def motor_write(self, command):
        self.active_motor_command = command
        self.motor_channel.write(command, self.motor_mask)

    # ========================================================
    # LEDs, Relay, and Buzzer
    # ========================================================

    def set_motion_led(self, state):
        self.set_output_bit(MOTION_LED_BIT, state)

    def set_buzzer(self, state):
        self.set_output_bit(BUZZER_BIT, state)

    def set_unlocked_led(self, state):
        self.set_output_bit(UNLOCKED_LED_BIT, state)

    def set_locked_led(self, state):
        self.set_output_bit(LOCKED_LED_BIT, state)

    def set_relay(self, state):
        self.set_output_bit(RELAY_BIT, state)

    def beep(self, duration_sec=0.15):
        period = 1.0 / BUZZER_FREQ_HZ
        half_period = period / 2.0
        end_time = time.monotonic() + duration_sec

        while time.monotonic() < end_time:
            self.set_buzzer(True)
            time.sleep(half_period)

            self.set_buzzer(False)
            time.sleep(half_period)

    def keypad_beep(self):
        self.beep(KEYPAD_BEEP_SEC)

    def success_buzzer(self):
        self.beep(0.12)
        time.sleep(0.08)
        self.beep(0.12)

    def denied_buzzer(self):
        self.beep(0.35)

    def lockout_buzzer(self):
        for _ in range(3):
            self.beep(0.25)
            time.sleep(0.15)

    # ========================================================
    # Motor Control
    # ========================================================

    def reset_motor_controller(self):
        print("Resetting motor controller...")

        self.motor_write(MOTOR_RESET)
        time.sleep(0.2)

        self.motor_write(MOTOR_STOP)
        time.sleep(0.2)

        print("Motor controller reset completed.")

    def start_open_lock(self):
        print("Opening lock...")

        self.set_locked_led(False)
        self.set_unlocked_led(True)

        self.motor_write(MOTOR_OPEN)
        self.motor_deadline = time.monotonic() + MOTOR_RUN_TIME_SEC
        self.state = SmartLockState.motor_opening

    def finish_open_lock(self):
        self.motor_write(MOTOR_STOP)

        print("Lock opened.")

        initial_raw = self.read_door_sensor()
        initial_state = "CLOSED" if initial_raw == 0 else "OPEN"
        print("Door state after unlocking: raw = {} | state = {}".format(
            initial_raw,
            initial_state
        ))

        print("Waiting for the door to open...")
        self.start_door_wait(
            state=SmartLockState.waiting_for_door_open,
            target_value=1,
            timeout_sec=DOOR_OPEN_TIMEOUT_SEC,
            stable_time_sec=DOOR_STABLE_TIME_SEC
        )

    def start_close_lock(self):
        if not self.is_door_closed():
            print("Door reopened before locking.")
            print("Waiting for the door to close again...")
            self.set_locked_led(False)
            self.set_unlocked_led(True)
            self.start_door_wait(
                state=SmartLockState.waiting_for_door_close,
                target_value=DOOR_CLOSED_VALUE,
                timeout_sec=DOOR_CLOSE_TIMEOUT_SEC,
                stable_time_sec=0.50
            )
            return

        print("Closing lock...")

        self.set_unlocked_led(False)
        self.set_locked_led(True)

        # Prepare the motor FSM for a new command after the opening cycle.
        # Sequence: STOP -> RESET -> STOP -> CLOSE.
        self.close_sequence = [
            (MOTOR_STOP, 0.10, ""),
            (MOTOR_RESET, 0.20, ""),
            (MOTOR_STOP, 0.20, ""),
            (MOTOR_CLOSE, MOTOR_RUN_TIME_SEC, "Sending MOTOR_CLOSE command..."),
            (MOTOR_STOP, 0.10, "")
        ]
        self.close_step_index = 0
        self.state = SmartLockState.motor_closing
        self.run_next_close_step()

    def run_next_close_step(self):
        if self.close_step_index >= len(self.close_sequence):
            print("Lock closed.")
            self.state = SmartLockState.locked
            self.close_sequence = []
            self.close_step_index = 0
            return

        command, duration_sec, message = self.close_sequence[self.close_step_index]

        if command == MOTOR_CLOSE and not self.is_door_closed():
            print("Door reopened before MOTOR_CLOSE.")
            print("Waiting for the door to close again...")
            self.motor_write(MOTOR_STOP)
            self.set_locked_led(False)
            self.set_unlocked_led(True)
            self.start_door_wait(
                state=SmartLockState.waiting_for_door_close,
                target_value=DOOR_CLOSED_VALUE,
                timeout_sec=DOOR_CLOSE_TIMEOUT_SEC,
                stable_time_sec=0.50
            )
            return

        if message:
            print(message)

        self.motor_write(command)
        self.motor_deadline = time.monotonic() + duration_sec
        self.close_step_index += 1

    # ========================================================
    # Door State Machine
    # ========================================================

    def unlock_then_auto_close(self):
        """Start a non-blocking 0 -> 1 -> 0 door cycle, then lock."""
        if self.is_access_cycle_active():
            print("Access cycle is already active. New unlock request ignored.")
            return

        self.start_open_lock()

    def is_access_cycle_active(self):
        return self.state != SmartLockState.locked

    def start_door_wait(self, state, target_value, timeout_sec, stable_time_sec):
        self.state = state
        self.door_wait_target = target_value
        self.door_wait_started_at = time.monotonic()
        self.door_wait_timeout = timeout_sec
        self.door_wait_stable_time = stable_time_sec
        self.door_wait_stable_since = None
        self.door_wait_last_value = None

    def poll_door_wait(self):
        now = time.monotonic()

        if now - self.door_wait_started_at >= self.door_wait_timeout:
            return False

        raw_value = self.read_door_sensor()

        if raw_value != self.door_wait_last_value:
            state = "CLOSED" if raw_value == 0 else "OPEN"
            print("Door sensor changed: raw = {} | state = {}".format(raw_value, state))
            self.door_wait_last_value = raw_value

        if raw_value == self.door_wait_target:
            if self.door_wait_stable_since is None:
                self.door_wait_stable_since = now
            elif now - self.door_wait_stable_since >= self.door_wait_stable_time:
                return True
        else:
            self.door_wait_stable_since = None

        return None

    def start_lock_delay(self):
        print("Door closing confirmed.")
        print("Waiting {} seconds before locking...".format(
            int(DOOR_LOCK_DELAY_SEC)
        ))

        self.lock_delay_until = time.monotonic() + DOOR_LOCK_DELAY_SEC
        self.state = SmartLockState.locking_delay

    def update_lock_delay(self):
        if self.read_door_sensor() != DOOR_CLOSED_VALUE:
            print("Door reopened during the locking delay.")
            print("Waiting for the door to close again...")
            self.start_door_wait(
                state=SmartLockState.waiting_for_door_close,
                target_value=DOOR_CLOSED_VALUE,
                timeout_sec=DOOR_CLOSE_TIMEOUT_SEC,
                stable_time_sec=0.50
            )
            return

        if time.monotonic() >= self.lock_delay_until:
            print("Door remained closed for {} seconds.".format(int(DOOR_LOCK_DELAY_SEC)))
            self.start_close_lock()

    def handle_door_open_timeout(self):
        print("Door was not opened within the allowed time.")
        print("Closing the lock automatically because the door remained closed.")

        self.write_log(
            source="DOOR_SENSOR",
            result="DOOR_NOT_OPENED",
            confidence="",
            system_state="UNLOCKED",
            image_path=""
        )

        if self.is_door_closed():
            self.start_close_lock()
        else:
            self.handle_door_left_open()
            self.state = SmartLockState.locked

    def handle_door_left_open(self):
        print("Door remained open beyond the allowed time.")
        print("The lock will remain open to avoid locking while the door is open.")

        self.write_log(
            source="DOOR_SENSOR",
            result="DOOR_LEFT_OPEN",
            confidence="",
            system_state="DOOR_OPEN",
            image_path=""
        )

        self.send_telegram_message(
            "Smart Lock Alert\n"
            "The door remained open for more than {} seconds\n"
            "The lock was not closed\n"
            "Time: {}".format(
                int(DOOR_CLOSE_TIMEOUT_SEC),
                now_string()
            )
        )

    def update_access_cycle(self):
        now = time.monotonic()

        if self.state == SmartLockState.motor_opening:
            if now >= self.motor_deadline:
                self.finish_open_lock()
            return

        if self.state == SmartLockState.waiting_for_door_open:
            door_opened = self.poll_door_wait()

            if door_opened is True:
                print("Door opening confirmed.")
                self.hold_open_until = time.monotonic() + 0.75
                self.state = SmartLockState.holding_door_open
            elif door_opened is False:
                self.handle_door_open_timeout()

            return

        if self.state == SmartLockState.holding_door_open:
            if now >= self.hold_open_until:
                print("Waiting for the door to close...")
                self.start_door_wait(
                    state=SmartLockState.waiting_for_door_close,
                    target_value=0,
                    timeout_sec=DOOR_CLOSE_TIMEOUT_SEC,
                    stable_time_sec=0.50
                )
            return

        if self.state == SmartLockState.waiting_for_door_close:
            door_closed = self.poll_door_wait()

            if door_closed is True:
                self.start_lock_delay()
            elif door_closed is False:
                self.handle_door_left_open()
                self.state = SmartLockState.locked

            return

        if self.state == SmartLockState.locking_delay:
            self.update_lock_delay()
            return

        if self.state == SmartLockState.motor_closing:
            if self.active_motor_command == MOTOR_CLOSE and not self.is_door_closed():
                print("Door opened while MOTOR_CLOSE was active.")
                print("Stopping motor and waiting for the door to close again...")
                self.motor_write(MOTOR_STOP)
                self.set_locked_led(False)
                self.set_unlocked_led(True)
                self.start_door_wait(
                    state=SmartLockState.waiting_for_door_close,
                    target_value=DOOR_CLOSED_VALUE,
                    timeout_sec=DOOR_CLOSE_TIMEOUT_SEC,
                    stable_time_sec=0.50
                )
                return

            if now >= self.motor_deadline:
                self.run_next_close_step()

    # ========================================================
    # Face Recognition Initialization
    # ========================================================

    def find_cascade_path(self):
        for path in CASCADE_PATHS:
            candidate = self.runtime_path(path)
            if candidate.is_file():
                return str(candidate)

        return None

    def load_face_recognition_models(self):
        cascade_path = self.find_cascade_path()

        if cascade_path is None:
            raise FileNotFoundError("Haar Cascade file was not found.")

        trainer_path = self.runtime_path(TRAINER_PATH)
        if not trainer_path.is_file():
            raise FileNotFoundError(f"LBPH trainer file was not found: {trainer_path}")

        print("Loading Haar Cascade face detector...")
        self.face_detector = cv2.CascadeClassifier(cascade_path)

        if self.face_detector.empty():
            raise RuntimeError("Failed to load Haar Cascade.")

        print("Face detector loaded successfully.")

        print("Loading LBPH face recognition model...")
        self.face_recognizer = cv2.face.LBPHFaceRecognizer_create()
        self.face_recognizer.read(str(trainer_path))

        print("LBPH model loaded successfully.")

    # ========================================================
    # Camera and Face Recognition Functions
    # ========================================================

    def capture_image(self, image_path=CAPTURED_IMAGE_PATH):
        print("Opening camera...")

        cap = None
        output_path = self.runtime_path(image_path)

        try:
            cap = cv2.VideoCapture(0)

            if not cap.isOpened():
                print("Camera 0 failed. Trying camera 1...")
                cap.release()
                cap = cv2.VideoCapture(1)

            if not cap.isOpened():
                print("Camera could not be opened.")
                return None

            time.sleep(0.5)

            ret, frame = cap.read()

            if not ret or frame is None:
                print("Image capture failed.")
                return None

            cv2.imwrite(str(output_path), frame)

            print("Image captured successfully:", str(output_path))

            return str(output_path)

        finally:
            if cap is not None:
                cap.release()

    def detect_and_recognize_face(self, image_path):
        if image_path is None:
            print("Image does not exist.")
            return False, None, None, ""

        image_file = Path(image_path)

        if not image_file.exists():
            print("Image does not exist.")
            return False, None, None, ""

        image = cv2.imread(str(image_file))

        if image is None:
            print("Failed to read image.")
            return False, None, None, ""

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        faces = self.face_detector.detectMultiScale(
            gray,
            scaleFactor=1.2,
            minNeighbors=5,
            minSize=(60, 60)
        )

        print("Number of faces detected:", len(faces))

        if len(faces) == 0:
            return False, None, None, str(image_file)

        best_confidence = None
        best_id = None

        for (x, y, w, h) in faces:
            face_roi = gray[y:y+h, x:x+w]

            predicted_id, confidence = self.face_recognizer.predict(face_roi)

            if best_confidence is None or confidence < best_confidence:
                best_confidence = confidence
                best_id = predicted_id

            label = "ID: {} Conf: {:.2f}".format(predicted_id, confidence)

            cv2.rectangle(image, (x, y), (x+w, y+h), (0, 255, 0), 2)

            cv2.putText(
                image,
                label,
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

        result_path = self.runtime_path(RESULT_IMAGE_PATH)
        cv2.imwrite(str(result_path), image)

        print("Best predicted ID:", best_id)
        print("Best confidence:", best_confidence)
        print("Result image saved:", str(result_path))

        return True, best_id, best_confidence, str(result_path)

    def is_authorized_face(self, predicted_id, confidence):
        return predicted_id == AUTHORIZED_USER_ID and confidence <= CONFIDENCE_THRESHOLD

    # ========================================================
    # Keypad Functions
    # ========================================================

    def read_keypad_status(self):
        return self.keypad_channel.read() & self.keypad_mask

    def decode_key(self, key_code):
        key_map = {
            0x0: "0",
            0x1: "1",
            0x2: "2",
            0x3: "3",
            0x4: "4",
            0x5: "5",
            0x6: "6",
            0x7: "7",
            0x8: "8",
            0x9: "9",
            0xA: "A",
            0xB: "B",
            0xC: "C",
            0xD: "D",
            0xE: "*",
            0xF: "#"
        }

        return key_map.get(key_code, "?")

    def read_keypad_key(self):
        status = self.read_keypad_status()

        key_valid = (status >> 4) & 0x1
        key_code = status & 0xF

        current_time = time.monotonic()

        if key_valid == 1:
            key = self.decode_key(key_code)

            if key != self.last_keypad_key or (current_time - self.last_keypad_time) > KEYPAD_DEBOUNCE_SEC:
                self.last_keypad_key = key
                self.last_keypad_time = current_time
                return key

        else:
            self.last_keypad_key = None

        return None

    def handle_keypad(self):
        if self.is_access_cycle_active():
            return

        key = self.read_keypad_key()

        if key is None:
            return

        self.keypad_beep()

        if self.is_system_locked_out():
            remaining = self.get_lockout_remaining_time()
            print("System is locked out. Remaining time:", remaining, "seconds.")
            self.denied_buzzer()
            return

        if key == "*":
            self.keypad_buffer = ""
            print("Keypad buffer cleared.")
            return

        if key == "#":
            entered_password = self.keypad_buffer
            self.keypad_buffer = ""

            print("Keypad password submitted.")

            if entered_password == self.password:
                self.handle_access_granted(source="KEYPAD")
            else:
                self.handle_access_denied(source="KEYPAD")

            return

        if key.isdigit():
            self.keypad_buffer += key
            print("Current keypad buffer length:", len(self.keypad_buffer))

            if len(self.keypad_buffer) > 8:
                self.keypad_buffer = ""
                print("Keypad buffer cleared because it was too long.")

            return

        print("Unsupported key ignored:", key)

    # ========================================================
    # Security / Failed Attempts / Lockout
    # ========================================================

    def reset_security_state(self):
        self.failed_attempts = 0
        self.failed_face_recognition_count = 0
        self.lockout_until = 0

        print("Security state reset.")
        print("Failed attempts:", self.failed_attempts)
        print("Failed face recognition count:", self.failed_face_recognition_count)

    def is_system_locked_out(self):
        if self.lockout_until == 0:
            return False

        if time.monotonic() < self.lockout_until:
            return True

        print("Lockout expired. System returned to normal mode.")

        self.failed_attempts = 0
        self.failed_face_recognition_count = 0
        self.lockout_until = 0

        return False

    def get_lockout_remaining_time(self):
        if self.lockout_until == 0:
            return 0

        remaining = int(self.lockout_until - time.monotonic())
        return max(0, remaining)

    def register_failed_attempt(self, source="UNKNOWN", confidence="", image_path=""):
        if self.is_system_locked_out():
            remaining = self.get_lockout_remaining_time()

            print("Access attempt ignored. System is locked out.")
            print("Remaining lockout time:", remaining, "seconds.")

            self.denied_buzzer()

            self.write_log(
                source=source,
                result="IGNORED_LOCKOUT",
                confidence=confidence,
                system_state="LOCKOUT",
                image_path=image_path
            )

            return

        self.failed_attempts += 1

        print("Access denied.")
        print("Source:", source)
        print("Failed attempts:", self.failed_attempts, "/", MAX_FAILED_ATTEMPTS)

        self.denied_buzzer()

        self.write_log(
            source=source,
            result="ACCESS_DENIED",
            confidence=confidence,
            system_state="LOCKED",
            image_path=image_path
        )

        # Telegram message for main failed attempt.
        # For FACE, the photo is sent only after 3 failed face events in register_failed_face_recognition().
        self.send_telegram_message(
            "Smart Lock Alert\n"
            "Access denied\n"
            "Source: {}\n"
            "Failed attempts: {}/{}\n"
            "Time: {}".format(
                source,
                self.failed_attempts,
                MAX_FAILED_ATTEMPTS,
                now_string()
            )
        )

        # Do not send face photos here to avoid duplicate Telegram photos.
        if image_path and source != "FACE":
            self.send_telegram_photo(
                image_path,
                "Access denied - source: {}".format(source)
            )

        if self.failed_attempts >= MAX_FAILED_ATTEMPTS:
            print("Maximum failed attempts reached.")

            if source == "KEYPAD":
                print("Capturing image after 3 wrong keypad attempts...")

                captured_keypad_image = self.capture_image(KEYPAD_INTRUDER_IMAGE_PATH)

                if captured_keypad_image is not None:
                    print("Keypad intruder image captured:", captured_keypad_image)

                    self.write_log(
                        source="KEYPAD",
                        result="KEYPAD_INTRUDER_PHOTO_CAPTURED",
                        confidence="",
                        system_state="LOCKED",
                        image_path=captured_keypad_image
                    )

                    self.send_telegram_photo(
                        captured_keypad_image,
                        "Smart Lock Alert\n"
                        "3 wrong keypad attempts detected\n"
                        "Intruder photo captured\n"
                        "Time: {}".format(now_string())
                    )

                else:
                    print("Failed to capture keypad intruder image.")

                    self.send_telegram_message(
                        "Smart Lock Alert\n"
                        "3 wrong keypad attempts detected\n"
                        "Photo capture failed\n"
                        "Time: {}".format(now_string())
                    )

            self.lockout_until = time.monotonic() + LOCKOUT_TIME_SEC

            print("Security lockout activated.")
            print("System locked for", LOCKOUT_TIME_SEC, "seconds.")

            self.lockout_buzzer()

            self.write_log(
                source="SYSTEM",
                result="LOCKOUT",
                confidence="",
                system_state="LOCKOUT",
                image_path=image_path
            )

            self.send_telegram_message(
                "Smart Lock Security Lockout\n"
                "The system was locked for {} seconds\n"
                "Reason: {}\n"
                "Time: {}".format(
                    LOCKOUT_TIME_SEC,
                    source,
                    now_string()
                )
            )

    def register_failed_face_recognition(self, confidence="", image_path=""):
        if self.is_system_locked_out():
            remaining = self.get_lockout_remaining_time()

            print("Face recognition attempt ignored. System is locked out.")
            print("Remaining lockout time:", remaining, "seconds.")

            self.denied_buzzer()

            self.write_log(
                source="FACE",
                result="IGNORED_LOCKOUT",
                confidence=confidence,
                system_state="LOCKOUT",
                image_path=image_path
            )

            return

        self.failed_face_recognition_count += 1

        print("Face recognition failed.")
        print("Face failed count:", self.failed_face_recognition_count, "/", MAX_FAILED_FACE_RECOGNITION)

        self.denied_buzzer()

        # Every failed face recognition event is written to CSV only.
        # Telegram is sent only after 3 consecutive failed face recognition events.
        self.write_log(
            source="FACE",
            result="FACE_RECOGNITION_FAILED",
            confidence=confidence,
            system_state="LOCKED",
            image_path=image_path
        )

        if self.failed_face_recognition_count >= MAX_FAILED_FACE_RECOGNITION:
            print("Three failed face recognition events counted as one failed access attempt.")
            print("Sending Telegram alert only now.")

            if image_path:
                self.send_telegram_photo(
                    image_path,
                    "Smart Lock Alert\n"
                    "3 failed face recognition events detected\n"
                    "This is counted as 1 failed access attempt\n"
                    "Time: {}".format(now_string())
                )
            else:
                self.send_telegram_message(
                    "Smart Lock Alert\n"
                    "3 failed face recognition events detected\n"
                    "No image available\n"
                    "This is counted as 1 failed access attempt\n"
                    "Time: {}".format(now_string())
                )

            self.failed_face_recognition_count = 0

            self.register_failed_attempt(
                source="FACE",
                confidence=confidence,
                image_path=image_path
            )

    def register_successful_access(self, source="UNKNOWN", confidence="", image_path=""):
        print("Access granted.")
        print("Source:", source)

        self.reset_security_state()

        self.success_buzzer()

        self.write_log(
            source=source,
            result="ACCESS_GRANTED",
            confidence=confidence,
            system_state="UNLOCKED",
            image_path=image_path
        )

        self.send_telegram_message(
            "Smart Lock\n"
            "Access granted\n"
            "Source: {}\n"
            "Time: {}".format(
                source,
                now_string()
            )
        )

    # ========================================================
    # Access Handling
    # ========================================================

    def handle_access_granted(self, source="UNKNOWN", confidence="", image_path=""):
        if self.is_access_cycle_active():
            print("Access cycle is already active. New access grant ignored.")
            return

        self.register_successful_access(
            source=source,
            confidence=confidence,
            image_path=image_path
        )

        self.unlock_then_auto_close()

    def handle_access_denied(self, source="UNKNOWN", confidence="", image_path=""):
        self.register_failed_attempt(
            source=source,
            confidence=confidence,
            image_path=image_path
        )

    # ========================================================
    # PIR + Light + Camera + Face Recognition Handler
    # ========================================================

    def handle_motion_event(self):
        if self.is_access_cycle_active():
            return

        print("Motion detected by PIR.")

        if self.is_system_locked_out():
            remaining = self.get_lockout_remaining_time()

            print("Motion detected, but system is locked out.")
            print("Remaining lockout time:", remaining, "seconds.")

            self.denied_buzzer()

            self.write_log(
                source="PIR",
                result="IGNORED_LOCKOUT",
                confidence="",
                system_state="LOCKOUT",
                image_path=""
            )

            return

        relay_was_enabled = False
        image_path = None

        self.set_motion_led(True)

        try:
            if self.is_dark():
                print("Dark environment detected. Relay ON.")

                self.set_relay(True)
                relay_was_enabled = True

                time.sleep(RELAY_PRE_LIGHT_SEC)

            else:
                print("Light environment detected. Relay remains OFF.")

            image_path = self.capture_image(CAPTURED_IMAGE_PATH)

        finally:
            if relay_was_enabled:
                self.set_relay(False)
                print("Relay OFF.")

            self.set_motion_led(False)

        if image_path is None:
            print("No image was captured.")
            print("This is counted as one failed face recognition event.")

            self.register_failed_face_recognition(
                confidence="",
                image_path=""
            )

            return

        face_found, predicted_id, confidence, result_image_path = self.detect_and_recognize_face(image_path)

        if not face_found:
            print("No face detected in the captured image.")
            print("This is counted as one failed face recognition event.")

            self.register_failed_face_recognition(
                confidence="",
                image_path=image_path
            )

            return

        if self.is_authorized_face(predicted_id, confidence):
            self.handle_access_granted(
                source="FACE",
                confidence="{:.2f}".format(confidence),
                image_path=result_image_path
            )

        else:
            print("Unknown or unauthorized face detected.")
            print("This is counted as one failed face recognition event.")

            if result_image_path and Path(result_image_path).exists():
                try:
                    image = cv2.imread(result_image_path)
                    unknown_path = self.runtime_path(UNKNOWN_IMAGE_PATH)
                    cv2.imwrite(str(unknown_path), image)
                    alert_image_path = str(unknown_path)
                except Exception:
                    alert_image_path = result_image_path
            else:
                alert_image_path = image_path

            self.register_failed_face_recognition(
                confidence="{:.2f}".format(confidence),
                image_path=alert_image_path
            )

    # ========================================================
    # Main System Loop
    # ========================================================

    def run(self):
        print("==============================================")
        print("Starting Final Smart Lock System")
        print("==============================================")

        self.init_log_file()
        self.load_system_overlay()
        self.load_face_recognition_models()

        self.init_outputs()
        self.reset_motor_controller()

        print("System is ready.")
        print("Waiting for PIR motion or keypad input.")
        print("Use keypad password: configured password followed by #")
        print("Keypad rule: wrong keypad code = 1 failed attempt.")
        print("Keypad rule: 3 wrong codes = intruder photo + Telegram + lockout.")
        print("Face rule: 3 failed face recognition events = 1 failed attempt + Telegram.")
        print("Door sensor: MC-38 on AR2; 0 = CLOSED, 1 = OPEN.")
        print("Automatic closing: after the door opens and closes.")
        print("Initial door state:", self.get_door_state())
        print("Telegram alerts:", "ON" if self.telegram_enabled else "OFF")
        print("Stop the cell manually when finished.")
        print("==============================================")

        self.send_telegram_message(
            "Smart Lock System Started\nTime: {}".format(now_string())
        )

        last_motion_time = 0

        while True:
            try:
                self.update_access_cycle()

                if not self.is_access_cycle_active():
                    self.handle_keypad()

                    pir_value = self.read_pir()

                    if pir_value == 1:
                        current_time = time.monotonic()

                        if current_time - last_motion_time >= MOTION_COOLDOWN_SEC:
                            last_motion_time = current_time
                            self.handle_motion_event()
                        else:
                            time.sleep(0.1)

                time.sleep(0.05)

            except KeyboardInterrupt:
                print("System stopped manually by user.")
                self.safe_shutdown()
                break

            except Exception as e:
                print("System error:", e)

                try:
                    self.motor_write(MOTOR_STOP)
                except Exception as stop_error:
                    print("Motor stop warning:", stop_error)

                self.write_log(
                    source="SYSTEM",
                    result="ERROR",
                    confidence="",
                    system_state="ERROR",
                    image_path=""
                )

                self.send_telegram_message(
                    "Smart Lock System Error\n{}\nTime: {}".format(
                        str(e),
                        now_string()
                    )
                )

                time.sleep(1)

    # ========================================================
    # Safe Shutdown
    # ========================================================

    def safe_shutdown(self):
        print("Shutting down system safely...")

        try:
            self.motor_write(MOTOR_STOP)

            self.output_state = 0
            self.write_outputs()

        except Exception as e:
            print("Shutdown warning:", e)

        self.write_log(
            source="SYSTEM",
            result="SHUTDOWN",
            confidence="",
            system_state="SHUTDOWN",
            image_path=""
        )

        self.send_telegram_message(
            "Smart Lock System Stopped\nTime: {}".format(now_string())
        )

        print("System shutdown completed.")


# ============================================================
# 8. Start System
# ============================================================

def run_smart_lock_system():
    system = SmartLockSystem()
    system.run()


# When this file is executed directly as a .py script, start automatically.
# Inside Jupyter, __file__ is not defined, so the system is started from a
# separate notebook cell by calling run_smart_lock_system().
if __name__ == "__main__" and "__file__" in globals():
    run_smart_lock_system()


In [ ]:
# Start the smart-lock system
run_smart_lock_system()
